<a href="https://colab.research.google.com/github/haotb/Demo/blob/master/Lucy_invoice__V004.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

https://drive.google.com/drive/folders/1CCVOX-g5ZJtiRD3WU6yZHGd7PTes7WOj

https://docs.google.com/document/d/1P6aqn7vwEBF3B6rD1heaGPZdgcq_8Bnfmh4Nej90zHo/edit?tab=t.0

In [ ]:
def clean_up_work_dir():
  import os
  import glob

  # Define patterns for PDF and Excel files
  patterns = ['*.pdf', '*.xlsx', '*.xls']
  files_to_delete = []
  for pattern in patterns:
      files_to_delete.extend(glob.glob(pattern))

  # Iterate and delete each file
  for file_path in files_to_delete:
      try:
          os.remove(file_path)
          print(f"Deleted: {file_path}")
      except Exception as e:
          print(f"Error deleting {file_path}: {e}")

  if not files_to_delete:
      print("No matching files (.pdf, .xlsx, .xls) found to delete.")
#####################
clean_up_work_dir()

Deleted: INV-OpCo013-684484-213540008.pdf
Deleted: SYSCO_CENTRAL_TEXAS__INC__213540008.xlsx


In [ ]:
import os
import imaplib
import email
from email.header import decode_header

IMAP_HOST = "imap.mail.yahoo.com"
IMAP_PORT = 993

# Define a global variable to store the last fetched msg_data
last_msg_data = None

def search_inbox_subject(keyword):
  global last_msg_data # Declare intent to modify the global variable
  print("keyword = ",  keyword )
  # username = os.getenv("YAHOO_USERNAME")
  # password = os.getenv("YAHOO_PASSWORD")
  username = "tianbao_hao"
  password = "qosnuczkpdusakwa"   ## generate app pw  https://help.yahoo.com/kb/SLN15241.html
  print("username = ", username)
  print("password = ", password)

  with imaplib.IMAP4_SSL(IMAP_HOST, IMAP_PORT) as imap:
    imap.login(username, password)
    imap.select("INBOX")

    status, data = imap.search(None, f'(SUBJECT "{keyword}")')
    print("status = ", status)
    print("data = ", data)

    if status != "OK":
      raise RuntimeError(f"Search failed: {status} {data}")

    seq=0
    for msg_id in data[0].split():
      print("**** ", seq, msg_id)
      seq+=1
      status, msg_data = imap.fetch(msg_id, "(RFC822)")
      print("status = ", status)
      print("msg_data = ", msg_data)
      # Store the last fetched message data in the global variable
      last_msg_data = msg_data

    imap.logout()

########################
if __name__ == "__main__":
  print("entering __main__()")
  search_inbox_subject("HaoDev")


entering __main__()
keyword =  HaoDev
username =  tianbao_hao
password =  qosnuczkpdusakwa
status =  OK
data =  [b'9985']
****  0 b'9985'
status =  OK
msg_data =  [(b'9985 (FLAGS (\\Seen $NotJunk) UID 409753 RFC822 {733912}', b'Received: from 127.0.0.1\r\n by atlas-production.v2-mail-prod1-gq1.omega.yahoo.com pod-id atlas--production-gq1-8cdff965d-5lths.gq1.yahoo.com with HTTP; Wed, 24 Jun 2026 14:59:53 +0000\r\nReturn-Path: <tianbao_hao@yahoo.com>\r\nX-Originating-Ip: [74.6.130.40]\r\nReceived-SPF: pass (domain of yahoo.com designates 74.6.130.40 as permitted sender)\r\nAuthentication-Results: mta.yahoo.com;\r\n dkim=pass header.i=@yahoo.com header.s=s2048 arc_overridden_status=NOT_OVERRIDDEN;\r\n spf=pass smtp.mailfrom=yahoo.com arc_overridden_status=NOT_OVERRIDDEN;\r\n dmarc=pass(p=REJECT) header.from=yahoo.com arc_overridden_status=NOT_OVERRIDDEN;\r\nX-Apparently-To: tianbao_hao@yahoo.com; Wed, 24 Jun 2026 14:59:54 +0000\r\nX-YMailISG: NTILG10WLDvTq3g7W.p2AGYhGLToAOG.GYGrw1FE_Z38OD

In [ ]:
from email.header import decode_header
from email import message_from_bytes

def parse_email_message(msg_data):
    # msg_data is a tuple, the second element is the raw email content
    # It's usually a list of tuples, so we take the first element (the actual message data tuple)
    raw_email = msg_data[0][1]
    msg = message_from_bytes(raw_email)
    return msg

def get_email_body(msg):
    body = ""
    if msg.is_multipart():
        for part in msg.walk():
            content_type = part.get_content_type()
            content_disposition = str(part.get_content_maintype())
            try:
                # Get the email body (plain text or HTML)
                if content_type == 'text/plain' and 'attachment' not in content_disposition:
                    body = part.get_payload(decode=True).decode()
                    break
                elif content_type == 'text/html' and 'attachment' not in content_disposition:
                    body = part.get_payload(decode=True).decode()
                    break
            except:
                pass
    else:
        try:
            body = msg.get_payload(decode=True).decode()
        except:
            pass
    return body

def get_email_attachments(msg):
    attachments = []
    for part in msg.walk():
        if part.get_content_maintype() == 'multipart':
            continue
        if part.get('Content-Disposition') is None:
            continue
        filename = part.get_filename()
        if filename:
            # Decode filename to handle encoded characters
            decoded_header = decode_header(filename)
            decoded_filename = ""
            for value, charset in decoded_header:
                if isinstance(value, bytes):
                    try:
                        # Try to decode using the specified charset, or utf-8 as a fallback
                        decoded_filename += value.decode(charset or 'utf-8')
                    except (UnicodeDecodeError, TypeError):
                        # Fallback for decoding issues
                        decoded_filename += value.decode('latin-1', errors='replace')
                else:
                    decoded_filename += value

            # Convert to ASCII, replacing unrepresentable characters
            ascii_filename = decoded_filename.encode('ascii', errors='replace').decode('ascii')

            attachments.append({
                'filename': ascii_filename,
                'payload': part.get_payload(decode=True)
            })
    return attachments

# Use the global variable last_msg_data from the previous cell execution

def extract_attachment():
  if 'last_msg_data' in globals() and last_msg_data is not None:
      # Parse the message
      email_message = parse_email_message(last_msg_data)

      # Extract the body
      email_body = get_email_body(email_message)
      print("\n--- Email Body ---")
      print(email_body[:500]) # Print first 500 characters of the body

      # Extract attachments
      email_attachments = get_email_attachments(email_message)
      print("\n--- Email Attachments ---")
      if email_attachments:
          for attachment in email_attachments:
              print(f"Filename: {attachment['filename']}, Size: {len(attachment['payload'])} bytes")
              yield attachment
      else:
          print("No attachments found.")
  else:
      print("Error: last_msg_data is not available. Please run the previous cell (KNZjvjdhFWS3) first.")

###
for e in extract_attachment():
  attachment=e
  print("attachment = ", attachment['filename'])
##########



--- Email Body ---
 I am doing Python Development.

--- Email Attachments ---
Filename: INV-OpCo013-684484-213540008.pdf, Size: 530928 bytes
attachment =  INV-OpCo013-684484-213540008.pdf


In [ ]:
attachment['filename']

'INV-OpCo013-684484-213540008.pdf'

In [ ]:
# Save the attachment to a file so it can be processed

def save_attachment(attachment):
  with open(attachment['filename'], 'wb') as f:
      f.write(attachment['payload'])

  print(f"Saved {attachment['filename']} to disk.")
##############################
save_attachment(attachment)

Saved INV-OpCo013-684484-213540008.pdf to disk.


In [ ]:
!apt-get update && apt-get install -y poppler-utils
import subprocess
import os
from google.colab import ai

# Point to the confirmed file path
pdf_path = attachment['filename']

def extract_data_with_colab_ai(path):
    try:
        # Convert PDF to text
        raw_text = subprocess.check_output(['pdftotext', '-layout', path, '-']).decode('utf-8')
    except Exception as e:
        return f"Failed to read PDF text: {e}"

    prompt = f"""
    You are a data extraction specialist. Below is the text content of an invoice PDF.
    Please extract the following information and return it in a clear, structured JSON format:
    - Vendor Name
    - Invoice Number
    - Date
    - Line Items (list with keys: Description, Quantity, Unit Price, Item Cost)

    INVOICE CONTENT:
    {raw_text}
    """

    print("Sending text to Colab AI for extraction...")
    response = ai.generate_text(prompt)
    return response

if os.path.exists(pdf_path):
    extraction_results = extract_data_with_colab_ai(pdf_path)
    print("\n--- AI Extraction Results ---")
    print(extraction_results)
else:
    print(f"Error: {pdf_path} not found.")

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:2 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:3 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Fetched 3,917 B in 2s (2,010 B/s)
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
poppler-utils is already the newest version (22.02

In [ ]:
# Purpose:
# Track weekly inventory, what was purchased, from what vendor, for what price
# It can use the “Inventory List” of all the items we buy for reference
# Output is data in a table with rows being each item in the list above, and columns representing an invoice from a vendor
# Each invoice should have a header of a date (the Monday of that week) and the vendor. Have 3 columns: pack/size, quantity bought, unit price, total price

# My idea for flow:
# Weekly, it pulls my email and searches for these emails/files
# Opens file, extracts the data, matches it up to the right row in Inventory List



In [ ]:
import json
import re

# This cell now safely waits for extraction_results to exist
if 'extraction_results' in globals():
    # 1. Extract the raw JSON string from the Markdown block
    json_match = re.search(r'```json\s*([\s\S]*?)\s*```', extraction_results)

    if json_match:
        json_content = json_match.group(1)
    else:
        json_content = extraction_results.strip()

    # 2. Convert the string into a dictionary
    extracted_data_dict = json.loads(json_content)

    # 3. Verify the result
    print(f"Dictionary created successfully. Type: {type(extracted_data_dict)}")
    display(extracted_data_dict)
else:
    print("Error: extraction_results not found. Please run the AI extraction cell (f4fed712) first.")

Dictionary created successfully. Type: <class 'dict'>


{'Vendor Name': 'SYSCO CENTRAL TEXAS, INC.',
 'Invoice Number': '213540008',
 'Date': '6/22/26',
 'Line Items': [{'Description': 'BKRSIMP BUN HAMBURGER BRIOCHE 4IN',
   'Quantity': 1,
   'Unit Price': 44.65,
   'Item Cost': 44.65},
  {'Description': 'HUYFONG SAUCE CHILI HOT SRIRACHA',
   'Quantity': 1,
   'Unit Price': 43.75,
   'Item Cost': 43.75},
  {'Description': 'HELLMAN MAYONNAISE REAL',
   'Quantity': 1,
   'Unit Price': 93.79,
   'Item Cost': 93.79},
  {'Description': 'SYS CLS SUGAR GRANULATED XFINE CANE',
   'Quantity': 1,
   'Unit Price': 38.55,
   'Item Cost': 38.55},
  {'Description': 'LABELLA PASTA MACARONI ELBOW LRG',
   'Quantity': 1,
   'Unit Price': 29.95,
   'Item Cost': 29.95},
  {'Description': 'UPPER BREAD CRUMB PANKO NON GMO',
   'Quantity': 1,
   'Unit Price': 34.95,
   'Item Cost': 34.95},
  {'Description': 'HAWNSUN DRINK PASSION FRUIT LILIKOI',
   'Quantity': 1,
   'Unit Price': 34.16,
   'Item Cost': 34.16},
  {'Description': 'HAWNSUN DRINK FRUIT PASS-O-GUAVA'

In [ ]:
import pandas as pd
import json

# Ensure we have the data dictionary
if 'extracted_data_dict' in globals():
    data = extracted_data_dict

    # The line items are in a list, but we want vendor/invoice info on every row
    rows = []
    vendor = data.get('Vendor Name', 'N/A')
    inv_num = data.get('Invoice Number', 'N/A')
    date = data.get('Date', 'N/A')

    for item in data.get('Line Items', []):
        rows.append({
            'Vendor Name': vendor,
            'Invoice Number': inv_num,
            'Date': date,
            'Line Item Description': item.get('Description', 'N/A'),
            'Quantity': item.get('Quantity', 0),
            'Unit Price': item.get('Unit Price', 0.0),
            'Item Cost': item.get('Item Cost', 0.0)
        })

    # Create DataFrame
    invoice_df = pd.DataFrame(rows)

    # Display the DataFrame
    display(invoice_df)
else:
    print("Error: extracted_data not found. Please run the extraction cell first.")

,Vendor Name,Invoice Number,Date,Line Item Description,Quantity,Unit Price,Item Cost
0,"SYSCO CENTRAL TEXAS, INC.",213540008,6/22/26,BKRSIMP BUN HAMBURGER BRIOCHE 4IN,1,44.65,44.65
1,"SYSCO CENTRAL TEXAS, INC.",213540008,6/22/26,HUYFONG SAUCE CHILI HOT SRIRACHA,1,43.75,43.75
2,"SYSCO CENTRAL TEXAS, INC.",213540008,6/22/26,HELLMAN MAYONNAISE REAL,1,93.79,93.79
3,"SYSCO CENTRAL TEXAS, INC.",213540008,6/22/26,SYS CLS SUGAR GRANULATED XFINE CANE,1,38.55,38.55
4,"SYSCO CENTRAL TEXAS, INC.",213540008,6/22/26,LABELLA PASTA MACARONI ELBOW LRG,1,29.95,29.95
5,"SYSCO CENTRAL TEXAS, INC.",213540008,6/22/26,UPPER BREAD CRUMB PANKO NON GMO,1,34.95,34.95
6,"SYSCO CENTRAL TEXAS, INC.",213540008,6/22/26,HAWNSUN DRINK PASSION FRUIT LILIKOI,1,34.16,34.16
7,"SYSCO CENTRAL TEXAS, INC.",213540008,6/22/26,HAWNSUN DRINK FRUIT PASS-O-GUAVA,1,37.67,37.67
8,"SYSCO CENTRAL TEXAS, INC.",213540008,6/22/26,SYS IMP TOWEL ROLL COMP360 NAT 8,1,45.69,45.69
9,"SYSCO CENTRAL TEXAS, INC.",213540008,6/22/26,SYS IMP NAPKIN DISP COMP360 1PLY NAT,1,57.99,57.99


In [ ]:
#

def save_invoice_to_excel(invoice_df, vendor, inv_num):
  import pandas as pd
  import re

  # Create a clean filename by concatenating vendor and invoice number
  # Removing non-alphanumeric characters for a safe filename
  clean_vendor = re.sub(r'[^a-zA-Z0-9]', '_', vendor)
  fn = f"{clean_vendor}_{inv_num}.xlsx"

  # Export the DataFrame to the new dynamic filename
  invoice_df.to_excel(fn, index=False)

  print(f'Successfully created {fn}')
  return fn
###
invoice_spreadsheet_fn = save_invoice_to_excel(invoice_df, vendor, inv_num)
invoice_spreadsheet_fn

Successfully created SYSCO_CENTRAL_TEXAS__INC__213540008.xlsx


'SYSCO_CENTRAL_TEXAS__INC__213540008.xlsx'

In [ ]:
import json
import re

# 1. Extract JSON from the markdown extraction_results
if 'extraction_results' in globals():
    json_match = re.search(r'```json\s*([\s\S]*?)\s*```', extraction_results)
    json_str = json_match.group(1) if json_match else extraction_results.strip()

    try:
        extracted_data = json.loads(json_str)
        line_items = extracted_data.get('Line Items', [])

        # 2. Calculate count and total
        num_items = len(line_items)
        total_cost = sum(item.get('Item Cost', 0) for item in line_items)

        print(f"--- Invoice Summary ---")
        print(f"Number of Line Items: {num_items}")
        print(f"Total Invoice Cost: ${total_cost:,.2f}")

    except Exception as e:
        print(f"Error parsing JSON data: {e}")
else:
    print("Error: extraction_results variable not found. Please run the AI extraction cell first.")

--- Invoice Summary ---
Number of Line Items: 16
Total Invoice Cost: $819.64


In [ ]:
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from email.mime.base import MIMEBase
from email import encoders
import os

def send_invoice_summary_email(recipient_email, total_amount, item_count, attachment_path=None):
    # Yahoo Mail SMTP settings
    SMTP_HOST = "smtp.mail.yahoo.com"
    SMTP_PORT = 465
    sender_email = "tianbao_hao@yahoo.com"
    sender_password = "qosnuczkpdusakwa"

    # Create the email message
    message = MIMEMultipart()
    message["From"] = sender_email
    message["To"] = recipient_email
    message["Subject"] = f"Invoice Summary - Total: ${total_amount:,.2f}"

    body = f"""Hello,

Here is the summary for the processed invoice:
- Total Invoice Cost: ${total_amount:,.2f}
- Number of Line Items: {item_count}

Please find the detailed Excel export attached.

Best regards,
Invoice Processor"""

    message.attach(MIMEText(body, "plain"))

    # Attach the Excel file
    if attachment_path and os.path.exists(attachment_path):
        with open(attachment_path, "rb") as attachment:
            part = MIMEBase("application", "octet-stream")
            part.set_payload(attachment.read())

        encoders.encode_base64(part)
        part.add_header(
            "Content-Disposition",
            f"attachment; filename={os.path.basename(attachment_path)}",
        )
        message.attach(part)

    try:
        # Connect and send
        with smtplib.SMTP_SSL(SMTP_HOST, SMTP_PORT) as server:
            server.login(sender_email, sender_password)
            server.sendmail(sender_email, recipient_email, message.as_string())
        print(f"Email with attachment successfully sent to {recipient_email}")
    except Exception as e:
        print(f"Failed to send email: {e}")

# --- Execution ---
target_email = "tianbao_hao@yahoo.com"

# Derive the filename as created in previous cells
import re
clean_vendor = re.sub(r'[^a-zA-Z0-9]', '_', vendor)
excel_fn = invoice_spreadsheet_fn

if 'total_cost' in globals() and 'num_items' in globals():
    send_invoice_summary_email(target_email, total_cost, num_items, excel_fn)
else:
    print("Error: Total cost or item count not found.")

Email with attachment successfully sent to tianbao_hao@yahoo.com


In [ ]:
import pandas as pd
import os

def match_invoice_to_inventory(invoice_df, inventory_path):
    """
    Matches items from the extracted invoice DataFrame to a master inventory list.
    """
    if not os.path.exists(inventory_path):
        print(f"Error: Inventory file '{inventory_path}' not found. Please upload it to the sidebar.")
        return None

    # Load master inventory
    if inventory_path.endswith('.csv'):
        inventory_df = pd.read_csv(inventory_path)
    else:
        inventory_df = pd.read_excel(inventory_path)

    # Performing a left merge based on the Description
    # Note: You may need to adjust the column names ('Line Item Description' vs 'Item Name')
    # to match your specific inventory file.
    merged_results = pd.merge(
        invoice_df,
        inventory_df,
        left_on='Line Item Description',
        right_on=inventory_df.columns[0], # Assuming first column is item name
        how='left'
    )

    print("Inventory matching complete.")
    return merged_results

# Example usage (uncomment once file is uploaded):
# inventory_file = 'Your_Master_Inventory.xlsx'
# results = match_invoice_to_inventory(invoice_df, inventory_file)
# display(results.head())

In [ ]:
# This cell previously contained 'sss', which caused a NameError because it is not defined in Python.

In [ ]:
# from google.colab import files
# import os

# # # 1. Prompt to upload the file from your local machine
# # print("Please upload 'pw.txt' from your C:\\temp\\ folder:")
# # uploaded = files.upload()
# uploaded="pw.txt"
# # 2. Check if the file was uploaded and read the content
# file_name = 'pw.txt'
# if file_name in uploaded:
#     with open(file_name, 'r') as f:
#         local_password = f.read().strip()
#     print(f"\nSuccessfully loaded password from {file_name}")
#     # You can now use 'local_password' in your SMTP or IMAP logic
# else:
#     print(f"\nError: {file_name} was not uploaded.")

In [ ]:
# Final Step: Match the extracted invoice data to your master inventory

inventory_filename = 'Your_Master_Inventory.xlsx' # Update this with your actual filename

if 'invoice_df' in globals():
    # Call the matching function defined in cell 615b3c12
    matched_results = match_invoice_to_inventory(invoice_df, inventory_filename)

    if matched_results is not None:
        print("Successfully matched invoice items to master inventory.")
        display(matched_results.head())
else:
    print("Error: No invoice data found. Please run the email retrieval and AI extraction cells first.")

Error: Inventory file 'Your_Master_Inventory.xlsx' not found. Please upload it to the sidebar.


In [ ]:
from google.colab import files
import os

print("Please upload your master inventory Excel or CSV file:")
uploaded = files.upload()

if uploaded:
    inventory_filename = list(uploaded.keys())[0]
    print(f"\nFile '{inventory_filename}' uploaded successfully.")

    # Automatically perform matching if invoice_df exists
    if 'invoice_df' in globals():
        matched_results = match_invoice_to_inventory(invoice_df, inventory_filename)
        if matched_results is not None:
            display(matched_results.head())
    else:
        print("Error: No invoice data found. Please ensure you have processed an invoice first.")

Please upload your master inventory Excel or CSV file:


In [ ]:
from google.colab import files
import os

print("Please upload your master inventory Excel or CSV file:")
uploaded = files.upload()

if uploaded:
    inventory_filename = list(uploaded.keys())[0]
    print(f"\nFile '{inventory_filename}' uploaded successfully.")

    # Now run the matching logic automatically
    if 'invoice_df' in globals():
        matched_results = match_invoice_to_inventory(invoice_df, inventory_filename)
        if matched_results is not None:
            display(matched_results.head())
    else:
        print("No invoice data found to match against.")